# Otonom Ajan Destekli Kestirimsel Analitik
## Banka Vadeli Mevduat Abonelik Tahmini

Bu projede Bank Marketing veri seti kullanılarak bir müşterinin vadeli mevduat ürününe abone olma olasılığı tahmin edilmektedir. Sistem; veri ön işleme, modelleme ve optimizasyon ile LLM tabanlı iş yorumu görevlerini üç ayrı ajan üzerinden uçtan uca yürütmektedir.

**Ajan mimarisi:**  
Ham Veri > Preprocessing Agent > Modeling Agent > Tahmin & Feature Importance > Insight Agent > İş Önerileri


## 1. Veri Setinin Yüklenmesi ve İlk İnceleme

Veri seti UCI Machine Learning Repository üzerinden alınmıştır. Öncelikle özellikler ve hedef değişken ayrılarak veri yapısı, eksik değerler ve sınıf dağılımı incelenmektedir.


In [44]:
import pandas as pd
from ucimlrepo import fetch_ucirepo

# Veri setini yükle
bank_marketing = fetch_ucirepo(id=222)

In [45]:
bank_marketing.variables

,name,role,type,demographic,description,units,missing_values
0,age,Feature,Integer,Age,NaN,NaN,no
1,job,Feature,Categorical,Occupation,"type of job (categorical: 'admin.','blue-colla...",NaN,no
2,marital,Feature,Categorical,Marital Status,"marital status (categorical: 'divorced','marri...",NaN,no
3,education,Feature,Categorical,Education Level,"(categorical: 'basic.4y','basic.6y','basic.9y'...",NaN,no
4,default,Feature,Binary,NaN,has credit in default?,NaN,no
5,balance,Feature,Integer,NaN,average yearly balance,euros,no
6,housing,Feature,Binary,NaN,has housing loan?,NaN,no
7,loan,Feature,Binary,NaN,has personal loan?,NaN,no
8,contact,Feature,Categorical,NaN,contact communication type (categorical: 'cell...,NaN,yes
9,day_of_week,Feature,Date,NaN,last contact day of the week,NaN,no


In [46]:
X = bank_marketing.data.features
y = bank_marketing.data.targets

print("X boyutu:", X.shape)
print("y boyutu:", y.shape)


X boyutu: (45211, 16)
y boyutu: (45211, 1)


In [47]:
import pandas as pd
import numpy as np

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

In [ ]:
%pip install langgraph langchain-openai

In [48]:
from typing import TypedDict, Any
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

In [49]:
import numpy as np

df = X.copy()
df["target"] = y.values.reshape(-1)

df.head()


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,target
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


In [50]:
# Veri setinin genel yapısı
print("Veri boyutu:", df.shape)

print("\n--- Veri Tipleri ---")
print(df.dtypes)

print("\n--- Eksik Değerler ---")
print(df.isnull().sum())

print("\n--- Target Dağılımı ---")
print(df["target"].value_counts())

print("\n--- Target Oranları (%) ---")
print(df["target"].value_counts(normalize=True) * 100)

Veri boyutu: (45211, 17)

--- Veri Tipleri ---
age            int64
job              str
marital          str
education        str
default          str
balance        int64
housing          str
loan             str
contact          str
day_of_week    int64
month            str
duration       int64
campaign       int64
pdays          int64
previous       int64
poutcome         str
target           str
dtype: object

--- Eksik Değerler ---
age                0
job              288
marital            0
education       1857
default            0
balance            0
housing            0
loan               0
contact        13020
day_of_week        0
month              0
duration           0
campaign           0
pdays              0
previous           0
poutcome       36959
target             0
dtype: int64

--- Target Dağılımı ---
target
no     39922
yes     5289
Name: count, dtype: int64

--- Target Oranları (%) ---
target
no     88.30152
yes    11.69848
Name: proportion, dtype: float64


In [51]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
age,45211.0,NaN,NaN,NaN,40.93621,10.618762,18.0,33.0,39.0,48.0,95.0
job,44923,11,blue-collar,9732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
marital,45211,3,married,27214,NaN,NaN,NaN,NaN,NaN,NaN,NaN
education,43354,3,secondary,23202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
default,45211,2,no,44396,NaN,NaN,NaN,NaN,NaN,NaN,NaN
balance,45211.0,NaN,NaN,NaN,1362.272058,3044.765829,-8019.0,72.0,448.0,1428.0,102127.0
housing,45211,2,yes,25130,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan,45211,2,no,37967,NaN,NaN,NaN,NaN,NaN,NaN,NaN
contact,32191,2,cellular,29285,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,45211.0,NaN,NaN,NaN,15.806419,8.322476,1.0,8.0,16.0,21.0,31.0


### Veri Kalitesi ve Sınıf Dağılımı

Hedef değişkende belirgin bir sınıf dengesizliği bulunmaktadır. Bu nedenle model değerlendirmesinde yalnızca accuracy yerine ROC-AUC, F1, Precision ve Recall metrikleri kullanılacaktır.


## 2. Feature Engineering ve Veri Hazırlığı

"duration" değişkeni görüşme tamamlandıktan sonra bilindiği için gerçek bir tahmin senaryosunda veri sızıntısına yol açabilir ve modelden çıkarılmıştır. Ayrıca müşterinin daha önce iletişime geçilip geçilmediğini temsil eden "previously_contacted" özelliği oluşturulmuştur.

"pdays" değişkeninin "-1" değerine sahip olmasının müşteriyle daha önce iletişime geçilmediği anlamına geldiğini ve "duration" değişkeninin veri sızıntısına yol açabileceği veri setinin açıklamalarında görülmektedir.

https://archive.ics.uci.edu/dataset/222/bank+marketing

In [52]:
# duration tahmin anında bilinmediği için modelden çıkarılır
X = df.drop(columns=["target", "duration"]).copy()

# Müşteriyle daha önce iletişime geçilip geçilmediğini gösteren yeni özellik
X["previously_contacted"] = (X["pdays"] != -1).astype(int)

# Target'ı binary formata dönüştür
y = df["target"].map({"no": 0, "yes": 1})

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (45211, 16)
y shape: (45211,)


In [ ]:
# Değişkenleri veri tiplerine göre ayır
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

In [56]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Kategorik ve sayısal değişkenler için preprocessing adımları
categorical_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="constant", fill_value="unknown")), ("encoder", OneHotEncoder(handle_unknown="ignore"))])

numerical_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])

preprocessor = ColumnTransformer(transformers=[("categorical", categorical_pipeline, categorical_cols), ("numerical", numerical_pipeline, numerical_cols)])

print("Preprocessing pipeline oluşturuldu.")

Preprocessing pipeline oluşturuldu.


In [57]:
from sklearn.model_selection import train_test_split, GridSearchCV


# Veriyi eğitim ve test olarak ayır
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

print("\nTrain target oranı:")
print(y_train.value_counts(normalize=True))


Train: (36168, 16)
Test: (9043, 16)

Train target oranı:
target
0    0.883018
1    0.116982
Name: proportion, dtype: float64


## 3. Agent 1 — Preprocessing & Outlier Detection

Preprocessing Agent; eksik değerleri işler, kategorik değişkenleri One-Hot Encoding ile dönüştürür, sayısal değişkenleri ölçekler ve IQR yöntemiyle uç değerleri raporlar. Uç değerler otomatik olarak silinmemiştir; bazı değişkenlerde bu değerler gerçek müşteri davranışını veya özel kodlanmış durumları temsil edebilir.


In [58]:
class PreprocessingAgent:
    """Veri ön işleme ve outlier analizini yöneten ajan."""

    def __init__(self, preprocessor):
        self.preprocessor = preprocessor
        self.report = {}

    def detect_outliers(self, X):
        numerical_data = X.select_dtypes(include=["number"])
        outlier_counts = {}

        for col in numerical_data.columns:
            q1 = numerical_data[col].quantile(0.25)
            q3 = numerical_data[col].quantile(0.75)
            iqr = q3 - q1

            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            outlier_counts[col] = int(((numerical_data[col] < lower_bound) | (numerical_data[col] > upper_bound)).sum())

        return outlier_counts

    def run(self, X_train, X_test):
        self.report["missing_values_before"] = int(X_train.isnull().sum().sum())
        self.report["outlier_counts"] = self.detect_outliers(X_train)

        X_train_processed = self.preprocessor.fit_transform(X_train)
        X_test_processed = self.preprocessor.transform(X_test)

        self.report["train_samples"] = X_train.shape[0]
        self.report["test_samples"] = X_test.shape[0]
        self.report["original_features"] = X_train.shape[1]
        self.report["processed_features"] = X_train_processed.shape[1]

        return X_train_processed, X_test_processed, self.report

In [59]:
preprocessing_agent = PreprocessingAgent(preprocessor)
X_train_processed, X_test_processed, preprocessing_report = preprocessing_agent.run(X_train, X_test)

print("Preprocessing tamamlandı.\n")
for key, value in preprocessing_report.items():
    print(f"{key}: {value}")


Preprocessing tamamlandı.

missing_values_before: 41691
outlier_counts: {'age': 394, 'balance': 3770, 'day_of_week': 0, 'campaign': 2465, 'pdays': 6584, 'previous': 6584, 'previously_contacted': 6584}
train_samples: 36168
test_samples: 9043
original_features: 16
processed_features: 51


## 4. Model Karşılaştırması

İlk aşamada Logistic Regression ve Random Forest modelleri aynı test seti üzerinde karşılaştırılmıştır. Sınıf dengesizliği nedeniyle her iki modelde de **class_weight="balanced"** kullanılmıştır.


In [60]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

# Karşılaştırılacak modeller
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=200, random_state=42, n_jobs=-1)
}

model_results = {}

for name, model in models.items():
    # Modeli eğit
    model.fit(X_train_processed, y_train)

    # Tahminleri üret
    y_pred = model.predict(X_test_processed)
    y_prob = model.predict_proba(X_test_processed)[:, 1]

    # Performans metriklerini kaydet
    model_results[name] = {
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "F1": f1_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred)
    }

results_df = pd.DataFrame(model_results).T
results_df

,ROC-AUC,F1,Precision,Recall
Logistic Regression,0.772237,0.373620,0.266667,0.623819
Random Forest,0.791082,0.434292,0.475281,0.399811


### Model Karşılaştırması Sonuçlarının Değerlendirilmesi

Random Forest modeli ROC-AUC ve F1 açısından Logistic Regression'dan daha iyi performans göstermiştir. Logistic Regression daha yüksek Recall üretse de Precision değeri daha düşüktür. Bu nedenle optimizasyon aşamasında Random Forest seçilmiştir.


## 5. Agent 2 — Model Kurma & Optimizasyon

Modeling Agent, Random Forest için GridSearchCV kullanarak hiperparametre optimizasyonu gerçekleştirir. Seçim kriteri 5-fold cross-validation ROC-AUC skorudur. Ardından en iyi model test setinde ROC-AUC, F1, Precision ve Recall ile değerlendirilir.


In [62]:
class ModelingAgent:
    """Model optimizasyonu, tahmin ve performans değerlendirmesini yöneten ajan."""

    def __init__(self):
        self.best_model = None
        self.best_params = None
        self.metrics = {}

    def run(self, X_train, X_test, y_train, y_test):
        param_grid = {
            "n_estimators": [100, 200],
            "max_depth": [10, 20, None],
            "min_samples_split": [2, 5],
            "class_weight": ["balanced"]
        }

        grid_search = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1), param_grid=param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
        grid_search.fit(X_train, y_train)

        self.best_model = grid_search.best_estimator_
        self.best_params = grid_search.best_params_

        y_pred = self.best_model.predict(X_test)
        y_prob = self.best_model.predict_proba(X_test)[:, 1]

        self.metrics = {
            "ROC-AUC": roc_auc_score(y_test, y_prob),
            "F1": f1_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred),
            "Recall": recall_score(y_test, y_pred)
        }

        return self.best_model, self.best_params, self.metrics


In [63]:
modeling_agent = ModelingAgent()

best_model, best_params, model_metrics = modeling_agent.run(X_train_processed, X_test_processed, y_train, y_test)

print("En iyi parametreler:", best_params)

print("\nModel performansı:")
for metric, value in model_metrics.items():
    print(f"{metric}: {value:.4f}")

En iyi parametreler: {'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}

Model performansı:
ROC-AUC: 0.8018
F1: 0.4731
Precision: 0.4188
Recall: 0.5435


### Optimize Edilmiş Model Sonuçları

Optimize edilmiş Random Forest modeli yaklaşık **0.802 ROC-AUC** ve **0.473 F1** skoruna ulaşmıştır. Optimizasyon sonrasında Recall artarken Precision bir miktar düşmüştür; bu durum modelin potansiyel aboneleri yakalama eğiliminin arttığını göstermektedir.


## 6. Feature Importance

Random Forest'ın feature importance değerleri modelin genel olarak hangi değişkenlerden daha fazla yararlandığını gösterir. Bu değerler tek bir müşteri tahmini için nedensel açıklama olarak yorumlanmamalıdır.


In [64]:
# Preprocessing sonrası feature isimlerini al
feature_names = preprocessing_agent.preprocessor.get_feature_names_out()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(10)

,feature,importance
45,numerical__balance,0.118679
44,numerical__age,0.111123
46,numerical__day_of_week,0.097758
47,numerical__campaign,0.053877
42,categorical__poutcome_success,0.051337
48,numerical__pdays,0.047669
27,categorical__contact_unknown,0.042750
22,categorical__housing_yes,0.027027
25,categorical__contact_cellular,0.026416
21,categorical__housing_no,0.026054


## 7. Agent 3 — LLM Insight & Executive Reporting

Insight Agent, modelin ürettiği tahmini, tahmin olasılığını, müşteri bilgilerini ve genel feature importance sonuçlarını bir LLM'e aktarır. Amaç teknik çıktıları iş diline çevirmek ve banka için uygulanabilir aksiyon önerileri üretmektir.

API anahtarı notebook içine kaydedilmez; çalışma sırasında gizli giriş alanından alınır.


In [21]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

OpenAI API Key:  ········


In [65]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0.2)

In [66]:
class InsightAgent:
    """Model tahminlerini LLM ile iş dilinde yorumlayan ajan."""

    def __init__(self, llm):
        self.llm = llm

    def run(self, prediction, probability, customer_data, top_features):
        prompt = f"""
        Bir banka pazarlama kampanyası için aşağıdaki müşteri tahminini yorumla.

        Tahmin: {"Abone olur" if prediction == 1 else "Abone olmaz"}
        Abonelik olasılığı: %{probability * 100:.1f}

        Müşteri bilgileri:
        {customer_data}

        Modelde genel olarak en önemli değişkenler:
        {top_features}

        Kısa ve profesyonel şekilde:
        1. Tahmin sonucunu yorumla.
        2. Tahmini etkileyebilecek ana faktörleri belirt.
        3. Banka için uygulanabilir 2 aksiyon öner.

        Not: Önemli değişkenler modelin genel feature importance değerleridir;
        bu nedenle tek müşteri için kesin nedensellik iddiasında bulunma.
        """

        response = self.llm.invoke(prompt)

        return response.content

In [67]:
sample_index = 0

sample_customer = X_test.iloc[sample_index]
sample_processed = X_test_processed[sample_index]

prediction = best_model.predict(sample_processed.reshape(1, -1))[0]
probability = best_model.predict_proba(sample_processed.reshape(1, -1))[0, 1]

top_features = feature_importance.head(5)[["feature", "importance"]].to_dict("records")

print("Tahmin:", prediction)
print("Abonelik olasılığı:", probability)
print("\nMüşteri:")
print(sample_customer)

Tahmin: 0
Abonelik olasılığı: 0.06463578960238128

Müşteri:
age                              40
job                     blue-collar
marital                     married
education                   primary
default                          no
balance                         640
housing                         yes
loan                            yes
contact                         NaN
day_of_week                       8
month                           may
campaign                          2
pdays                            -1
previous                          0
poutcome                        NaN
previously_contacted              0
Name: 1392, dtype: object


In [68]:
insight_agent = InsightAgent(llm)

report = insight_agent.run(
    prediction=prediction,
    probability=probability,
    customer_data=sample_customer.to_dict(),
    top_features=top_features
)

print(report)

1. **Tahmin yorumu:**  
   Model, müşterinin abone olma olasılığını **%6,5** olarak hesaplıyor ve bu nedenle **“abone olmaz”** tahmininde bulunuyor. Bu, müşterinin kesinlikle abone olmayacağı anlamına gelmez; yalnızca mevcut bilgilerle düşük eğilim gösterdiğini ifade eder.

2. **Tahmini etkileyebilecek ana faktörler:**  
   - **Bakiye (640):** Modelde en önemli değişkenlerden biri; müşterinin finansal profilini etkiliyor olabilir.  
   - **Yaş (40):** Abonelik eğilimini etkileyen önemli bir demografik faktör.  
   - **Görüşme günü ve kampanya temas sayısı:** Ayın 8. günü ve ikinci kampanya teması, yanıt olasılığını etkileyebilir.  
   - **Konut kredisi ve ihtiyaç kredisi bulunması:** Mevcut borçluluk, yeni bir bankacılık ürününe ilgiyi azaltabilir.  
   - **Geçmiş iletişim bilgilerinin bulunmaması:** `contact` ve `poutcome` verilerinin eksik olması tahmin belirsizliğini artırabilir.  
   
   Bu değerlendirmeler modelin **genel değişken önemlerine** dayanır; tek müşteri için kesin neden

In [69]:
class AgentState(TypedDict, total=False):
    X_train: Any
    X_test: Any
    y_train: Any
    y_test: Any

    X_train_processed: Any
    X_test_processed: Any
    preprocessing_report: dict

    best_model: Any
    best_params: dict
    model_metrics: dict

    prediction: int
    probability: float
    customer_data: dict
    top_features: list

    llm_report: str
    sample_index: int

## 8. End-to-End Agentic Pipeline

Son aşamada üç ajan tek bir akışta birbirine bağlanmaktadır. Preprocessing Agent veriyi hazırlar, Modeling Agent modeli optimize ederek tahmini üretir ve Insight Agent sonucu iş perspektifinden yorumlar.


In [70]:
def preprocessing_node(state: AgentState) -> dict:
    """Preprocessing Agent'ı çalıştırır ve sonuçları state'e aktarır."""

    X_train_processed, X_test_processed, report = preprocessing_agent.run(
        state["X_train"],
        state["X_test"]
    )

    return {
        "X_train_processed": X_train_processed,
        "X_test_processed": X_test_processed,
        "preprocessing_report": report
    }

In [71]:
def modeling_node(state: AgentState) -> dict:
    """Modeling Agent'ı çalıştırır ve tahmin sonuçlarını state'e aktarır."""

    best_model, best_params, model_metrics = modeling_agent.run(
        state["X_train_processed"],
        state["X_test_processed"],
        state["y_train"],
        state["y_test"]
    )

    # Yorumlanacak müşteriyi seç
    sample_index = state["sample_index"]
    sample_customer = state["X_test"].iloc[sample_index]
    sample_processed = state["X_test_processed"][sample_index:sample_index + 1]

    # Tahmin ve olasılık
    prediction = best_model.predict(sample_processed)[0]
    probability = best_model.predict_proba(sample_processed)[0, 1]

    # Modelin genel feature importance değerleri
    feature_names = preprocessing_agent.preprocessor.get_feature_names_out()
    feature_importance = pd.DataFrame({
        "feature": feature_names,
        "importance": best_model.feature_importances_
    }).sort_values("importance", ascending=False)

    top_features = feature_importance.head(5)[["feature", "importance"]].to_dict("records")

    return {
        "best_model": best_model,
        "best_params": best_params,
        "model_metrics": model_metrics,
        "prediction": prediction,
        "probability": probability,
        "customer_data": sample_customer.to_dict(),
        "top_features": top_features
    }

In [72]:
def insight_node(state: AgentState) -> dict:
    """Insight Agent'ı çalıştırır ve LLM raporunu state'e aktarır."""

    llm_report = insight_agent.run(
        prediction=state["prediction"],
        probability=state["probability"],
        customer_data=state["customer_data"],
        top_features=state["top_features"]
    )

    return {
        "llm_report": llm_report
    }

In [73]:
workflow = StateGraph(AgentState)

workflow.add_node("preprocessing", preprocessing_node)
workflow.add_node("modeling", modeling_node)
workflow.add_node("insight", insight_node)

workflow.set_entry_point("preprocessing")

workflow.add_edge("preprocessing", "modeling")
workflow.add_edge("modeling", "insight")
workflow.add_edge("insight", END)

app = workflow.compile()

In [74]:
initial_state = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "sample_index": 0
}

final_output = app.invoke(initial_state)

In [75]:
print("--- PREPROCESSING REPORT ---")
print(final_output["preprocessing_report"])

print("\n--- BEST PARAMETERS ---")
print(final_output["best_params"])

print("\n--- MODEL METRICS ---")
print(final_output["model_metrics"])

print("\n--- CUSTOMER PREDICTION ---")
print("Tahmin:", final_output["prediction"])
print(f"Abonelik olasılığı: %{final_output['probability'] * 100:.1f}")

print("\n--- LLM INSIGHT REPORT ---")
print(final_output["llm_report"])

--- PREPROCESSING REPORT ---
{'missing_values_before': 41691, 'outlier_counts': {'age': 394, 'balance': 3770, 'day_of_week': 0, 'campaign': 2465, 'pdays': 6584, 'previous': 6584, 'previously_contacted': 6584}, 'train_samples': 36168, 'test_samples': 9043, 'original_features': 16, 'processed_features': 51}

--- BEST PARAMETERS ---
{'class_weight': 'balanced', 'max_depth': 20, 'min_samples_split': 5, 'n_estimators': 200}

--- MODEL METRICS ---
{'ROC-AUC': 0.8018199293808215, 'F1': 0.4730563554092966, 'Precision': 0.4187909686817189, 'Recall': 0.5434782608695652}

--- CUSTOMER PREDICTION ---
Tahmin: 0
Abonelik olasılığı: %6.5

--- LLM INSIGHT REPORT ---
1. **Tahmin sonucu:**  
   Model, müşterinin abonelik ürününe katılmayacağını öngörüyor. **%6,5’lik olasılık düşük** olduğundan müşteri, öncelikli hedef kitlede değerlendirilmemeli; ancak bu sonuç kesin ret anlamına gelmez.

2. **Tahmini etkileyebilecek ana faktörler:**  
   - **Bakiye (640):** Modelde en yüksek öneme sahip değişken.  
   

## 9. Sonuç ve Değerlendirme

Bu projede üç ajanlı bir kestirimsel analitik sistemi oluşturulmuştur:

- **Preprocessing Agent:** Eksik değer işleme, encoding, scaling ve IQR tabanlı outlier analizi gerçekleştirmiştir.
- **Modeling Agent:** Random Forest modelini GridSearchCV ile optimize etmiş ve performans metriklerini üretmiştir.
- **Insight Agent:** Model tahminini ve genel feature importance sonuçlarını LLM aracılığıyla iş diline çevirmiş ve aksiyon önerileri üretmiştir.

Optimize edilmiş modelin ROC-AUC skoru yaklaşık **0.802**'dir. Sistem yalnızca tahmin üretmekle kalmayıp tahmin sonucunu karar vericiler için yorumlanabilir ve aksiyona dönüştürülebilir bir çıktıya çevirmektedir.

### Sınırlılıklar

Feature importance değerleri modelin genel davranışını açıklar; tek bir müşterinin tahminini yerel olarak açıklamaz. Ayrıca veri setindeki sınıf dengesizliği nedeniyle Precision ve Recall arasında bir denge bulunmaktadır. Gelecek çalışmalarda SHAP tabanlı yerel açıklamalar ve karar eşiği optimizasyonu eklenebilir.
